# Manchester City Forecasting Project

*An integrated review of the models, results, limitations, and next steps*

**Version 1 retrospective — September 2026**

This notebook brings together the reasoning behind the project's six modeling stages: constructing team performance states, defining a strength index, checking it against ClubElo, initializing promoted teams, updating strengths with Bayesian inference, and forecasting City matches.

The focus is on the decisions made along the way: which assumptions were tested, which components were retained, and where the evidence led to a simpler model. It also sets out a possible Version 2 for the whole Premier League. That extension remains a research proposal.

The numerical results below come from the saved outputs of the source notebooks listed in Appendix A. They use different targets, samples, and evaluation designs, so their metrics should be compared within the relevant experiment. Reviewing those outputs is not a fresh end-to-end reproduction of the project.

Each numbered topic is in a separate Markdown cell so that the text, equations, and tables can be edited individually in VS Code.


## Part I — How the project developed


### 1. The original aim


The project began with a broad question: could a score forecast combine team strength with information about playing style, recent performance, coaching changes, and squad reconstruction?

The intended model linked Bayesian strength estimates to a dynamic performance state, then translated that state into score probabilities. As the work progressed, several plausible additions failed to improve the historical comparisons. The operational version consequently settled on Bayesian strength difference and venue as inputs to two Poisson goal rates.

Coaching, midfield, and adaptation models remain part of the project as structural analyses. Their role is to describe possible changes and the assumptions behind them. They are not currently inputs to the operational score rates.


### 2. How model components were judged


A component can be interpretable, fit historical observations, and satisfy its mathematical assumptions without helping to predict a later match. The project therefore evaluated interpretation, numerical behavior, and predictive performance separately.

Football knowledge helped generate the hypotheses. Model comparisons determined whether the available evidence justified using them in the forecaster. When an extension did not improve those comparisons, its role was narrowed or it was left out of the operational model. This approach provided a practical stopping rule for a project with many possible directions.


## Part II — Opponent-adjusted team performance

*Source: team_strength.ipynb*


### 3. Building the first team representation


Raw football statistics reflect both a team's own performance and the conditions of the match. Opponent quality, venue, and ordinary match variation all affect the observed numbers.

The first notebook constructs a venue-specific team representation from four performance differences:

$$
\mathbf x=
\begin{bmatrix}
ShotDiff & SOTRateDiff & FinishingDiff & CornersDiff
\end{bmatrix}^{T}.
$$

This is the upstream strength-construction vector. The later City state replaces corners with possession; the two four-dimensional representations serve different purposes and should not be treated as identical.


### 4. Why these features were chosen


The first three variables distinguish shot generation, shot accuracy, and finishing. Corners provide an additional measure of attacking pressure. Using several stages of the attacking process allows teams with similar goal totals to have different performance profiles.

Each variable measures the team's advantage over its opponent in the same match. For example,

$$
ShotDiff=Shots_{\mathrm{team}}-Shots_{\mathrm{opponent}}.
$$

The four-variable choice is a deliberate simplification. It captures a manageable set of attacking dimensions without claiming to cover every aspect of football performance.


### 5. Adjusting for opponent strength


Opponent strength is standardized so that $q=0$ represents the reference population's average opponent. For team $i$, venue $v$, feature $k$, season $s$, and match $m$, the adjustment is

$$
z_{i,v,m,k}^{(s)}
=\alpha_{i,v,k}^{(s)}
+\beta_{v,k}^{(s)}q_{j(m)}^{(s)}
+\varepsilon_{i,v,m,k}^{(s)}.
$$

Here, $z$ is the standardized performance variable, $\alpha$ is a team-specific baseline, and $\beta$ is the shared opponent-response slope within the relevant league, venue, season, and feature.

Under this model, $\alpha$ estimates expected performance against an average-strength opponent. It provides a common reference for teams whose observed schedules differ.


### 6. Testing a nonlinear opponent response


The notebook compares a linear response with a quadratic alternative:

$$
z=\alpha_i+\beta q+\varepsilon,
\qquad
z=\alpha_i+\beta_1q+\beta_2q^2+\varepsilon.
$$

Repeated stratified five-fold cross-validation is used, with a 5% RMSE reduction required to retain the extra curvature. Across the sixteen league, venue, and feature comparisons, the reported relative gains range from approximately $-0.44\%$ to $1.22\%$. None reaches the threshold.

The linear response is therefore retained. This is a decision about the tested specification and sample, not a claim that football responses are universally linear. The fold-based comparison assesses generalization within this design; it is not itself a chronological next-season test.


### 7. Testing team-specific opponent sensitivities


Different clubs may respond differently to strong opponents, but estimating a separate slope from roughly nineteen home or away matches would leave considerable uncertainty. The notebook tests a partially pooled alternative:

$$
\beta_i=\mu_\beta+u_i,
\qquad
u_i\sim\mathcal N(0,\tau_\beta^2).
$$

This allows team slopes to differ while shrinking noisy estimates toward a shared mean. The cross-validation results provide little support for the extra variation. The only positive RMSE reduction, for EPL home shot difference, is approximately $0.0013\%$; the other cases are approximately equal or worse.

Version 1 retains team-specific baselines and shared opponent slopes. Team differences remain represented through baseline performance and residual covariance, while separately estimated response slopes are excluded.


### 8. What the opponent adjustment found


The strongest relationships occur in volume-related variables. In the 2021/22 EPL diagnostic season, the shot-difference slopes are approximately

$$
\widehat\beta_{\mathrm{Shot,Home}}=-0.434,
\qquad
\widehat\beta_{\mathrm{Shot,Away}}=-0.436.
$$

Corner difference also has a substantial negative association with opponent strength. Shot-on-target rate and finishing show weaker or less precisely estimated relationships.

This provides an early indication that volume and efficiency should not be assumed equally predictable. A similar distinction appears in the later City residual analysis, although the two stages use different models and targets.


### 9. Residual covariance and the instability index


After fitting the baseline and opponent adjustment, residuals are

$$
\widehat{\boldsymbol\varepsilon}_{i,v,m}
=\mathbf z_{i,v,m}
-\widehat{\boldsymbol\alpha}_{i,v}
-\widehat{\boldsymbol\beta}_{v}q_{j(m)}.
$$

Their covariance describes the size of unexplained variation and how the retained dimensions move together. The scalar instability measure is

$$
U_{i,v}^{(s)}
=\sqrt{\frac{1}{p}
\operatorname{tr}\!\left(\widehat\Sigma_{i,v}^{(s)}\right)}.
$$

Because the features are standardized, $U$ is the root-mean-square residual standard deviation across them. A lower value indicates performances closer to the fitted expectation; a higher value indicates greater residual variation.

Strength and instability describe different properties. A strong team may fluctuate considerably around a high baseline, while a weaker team may remain close to a low one.


### 10. What the bootstrap adds


The 2021/22 diagnostic analysis uses 5,000 replications of a fixed-design Mammen wild bootstrap. It examines uncertainty in opponent slopes, team baselines, and the instability index.

Shot and corner effects are comparatively well supported, while several efficiency effects and the magnitude of team instability are less precise. City's instability estimates illustrate this:

| Venue | Original $U$ | Bootstrap median | 95% bootstrap interval |
|---|---:|---:|---|
| Away | 0.7556 | 0.7186 | [0.5276, 0.9477] |
| Home | 0.9183 | 0.8699 | [0.6239, 1.1470] |

The home point estimate is larger, but the broad, overlapping marginal intervals indicate considerable uncertainty. A formal assessment of the home-away difference would use the bootstrap distribution of that difference, rather than interval overlap alone.

The notebook retains adjusted baselines, shared linear slopes, residual covariance, and $U$. Quadratic responses and team-specific slopes are excluded from the final adjustment.


## Part III — From team states to scalar strength

*Source: strength_index_validation.ipynb*


### 11. Why a scalar strength measure is needed


The Bayesian layer needs a compact initial strength coordinate. Each four-dimensional venue baseline is projected onto a scalar:

$$
S_{i,v}^{(s)}
=\mathbf w_v^{T}\widehat{\boldsymbol\alpha}_{i,v}^{(s)},
\qquad
w_k\geq0,\quad\sum_k w_k=1.
$$

Nonnegative weights preserve the interpretation that stronger performance in a retained dimension contributes positively to the index. Home and away strength remain separate at this stage.


### 12. The equal-weight reference


The initial projection assigns a weight of 0.25 to each component. It already agrees closely with venue-specific league performance: mean Spearman correlations are approximately 0.876 at home and 0.833 away.

This establishes a useful reference before weight optimization. The question becomes whether unequal weights improve ranking agreement enough to justify replacing a simple and interpretable projection.


### 13. Calibrating weights across seasons


Weights are calibrated separately by venue using leave-one-season-out validation. For each held-out season, the search uses the other four seasons and evaluates the selected weights on the omitted one.

Mean Spearman correlations increase to approximately 0.899 at home and 0.856 away, an improvement of about 0.024 for each venue when calculated from the unrounded outputs.

This is evidence of ranking agreement across held-out seasons. Because a fold can train on seasons later than its held-out season, it should not be described as a strictly forward-looking forecast experiment.


### 14. Checking sensitivity to the random search


The weight search is repeated with multiple random seeds. The broad pattern remains similar, leading to the following rounded multi-seed mean weights:

| Venue | Shot difference | SOT-rate difference | Finishing difference | Corner difference |
|---|---:|---:|---:|---:|
| Home | 0.297 | 0.287 | 0.346 | 0.070 |
| Away | 0.286 | 0.175 | 0.480 | 0.059 |

These weights define a projection fitted to the chosen ranking target. They are not causal shares of football ability: an away finishing weight near 0.48 does not mean that finishing constitutes 48% of a team's true quality.


### 15. Competitive tiers for interpretation


The scalar venue strengths are clustered to make their distribution easier to interpret. Candidate cluster counts from two to six are compared using inertia and silhouette scores; three tiers are retained for both venues: Lower, Competitive, and Elite.

The underlying model state remains continuous, $S\in\mathbb R$. Tier labels are used for interpretation and visualization, so teams within a tier can still have different strength values.


### 16. What this stage contributes


This stage produces continuous home and away strength measures with fixed projection weights. Clustering provides a way to describe the results but does not replace the numerical strengths used downstream.

The next question is how to combine the two venue measures while retaining information about home-away differences. That motivates the PCA stage.


## Part IV — Combining venues and checking against ClubElo

*Source: strength_index_pca_elo_validation.ipynb*


### 17. The common home-away signal


Home and away strength are standardized separately:

$$
Z_H=\frac{S_H-\mu_H}{\sigma_H},
\qquad
Z_A=\frac{S_A-\mu_A}{\sigma_A}.
$$

Their pooled correlation is approximately 0.7504. The positive association suggests a substantial shared strength component, with additional variation arising from venue asymmetry.


### 18. Interpreting the PCA coordinates


For these two standardized variables, the fitted principal components are essentially

$$
PC1=\frac{Z_H+Z_A}{\sqrt2},
\qquad
PC2=\frac{Z_H-Z_A}{\sqrt2},
$$

up to the arbitrary sign of each PCA axis. PC1 explains 87.52% of the joint variation and PC2 explains 12.48%.

The first component represents common overall strength; the second describes home-away asymmetry. In this two-variable setting, PCA is a transparent rotation into a shared direction and a venue contrast.


### 19. Agreement with ClubElo


ClubElo is introduced after constructing the strength index and PCA representation. It serves as an external comparison rather than an input to their fitted weights.

The primary benchmark averages each team's Elo over its actual EPL match dates. Missing values are not interpolated; the coverage audit retains 98 of 100 team-seasons. PC1's seasonal Pearson correlations with that benchmark are:

| Season | Pearson $r$ |
|---|---:|
| 2021/22 | 0.924 |
| 2022/23 | 0.827 |
| 2023/24 | 0.884 |
| 2024/25 | 0.918 |
| 2025/26 | 0.848 |

The median Pearson correlation is approximately 0.884, and the median Spearman correlation is approximately 0.856. PC2's relationship with Elo is weaker and less consistent.

The project adopts $S_{i,s}=PC1_{i,s}$ as unified strength and retains $V_{i,s}=PC2_{i,s}$ separately as venue asymmetry. The Elo agreement supports the interpretation of PC1 as strength, but does not by itself validate future score probabilities.


### 20. The timing limit of pooled PCA


Using all five completed seasons to construct a representation for later application is appropriate once those seasons are historical. A strict retrospective forecast, however, must fit scaling and PCA only on information available before its target period.

Otherwise, later feature distributions influence the coordinate system even when no target score is directly supplied to the predictor. This distinction between retrospective representation and prospective forecasting applies again to the midfield SVD.


## Part V — Initializing promoted clubs

*Source: E0_E1_bridge.ipynb*


### 21. Why the leagues need a bridge


A promoted club has no previous-season EPL strength under the current representation. Its Championship strength is measured against a different competitive reference population and cannot be inserted unchanged into the EPL scale.

The promotion bridge tests how much Championship information can support an EPL preseason estimate. Two versions appear in the project: an earlier bridge on the opponent-strength scale $q$, and a later bridge on the final unified scale $S$.


### 22. The preliminary opponent-strength bridge


The earlier analysis uses 33 promotion transitions and tests several mappings. One candidate is a constrained shrinkage model:

$$
q_{\mathrm{EPL}}=\mu+\lambda q_{\mathrm{EFL}},
\qquad 0\leq\lambda\leq1.
$$

Leave-one-promotion-cohort-out cross-validation selects $\lambda=0$, with $\mu\approx-0.971$. Under this design, the Championship measure does not add a reliable team-specific ranking signal after promotion.

The resulting common mean expresses limited information in the tested bridge. It does not imply that promoted clubs have identical true ability. Cohort-wise validation also differs from a purely chronological test when later cohorts can contribute to a training fold.


### 23. Adding summer investment to the bridge


Summer expenditure is tested as another possible predictor. The reported leave-one-cohort-out RMSE values are:

| Specification | CV RMSE |
|---|---:|
| Mean only | 0.6997 |
| Investment only | 0.7015 |
| Championship $q$ | 0.7240 |
| Championship $q$ + investment | 0.7240 |

Investment does not improve this comparison, so the preliminary bridge remains mean-only. This result concerns the aggregate financial measure, the promotion target, and the available sample.


### 24. The bridge on the final strength scale


The later bridge uses twelve promotion transitions for which both Championship and subsequent EPL strength can be reconstructed under the final $S$ specification. The raw cross-league correlation is approximately 0.107.

The constant model has cross-validated RMSE 1.1283, compared with 1.1577 for a linear mapping. The constant bridge is retained, with a full-sample promoted-team mean of

$$
\mu_{\mathrm{promoted}}=-1.613453.
$$

For the 2026/27 application, promoted clubs receive that common prior mean, while their prior variances differ through $P_{i,0}=Q_0M_i$. The historical promoted-team standard deviation of 1.024 describes heterogeneity; it is not added separately to that calibrated variance.


### 25. Keeping the two bridge stages distinct


| Stage | Coordinate | Promotion transitions | Retained center | Role |
|---|---|---:|---:|---|
| Preliminary bridge | Opponent strength $q$ | 33 | Approximately −0.971 | Earlier methodological development, with a Student-$t$ predictive representation |
| Final bridge | Unified strength $S$ | 12 | Approximately −1.613 | Preseason mean used by the final Bayesian system |

The smaller final sample reflects the shorter window available under a consistent unified-strength construction. The two means belong to different scales and should not be compared numerically as though they measured the same quantity.


## Part VI — Recursive Bayesian strength

*Source: bayesian_prior.ipynb*


### 26. Carrying information across seasons


The recursive model uses the previous season's posterior mean to initialize retained clubs:

$$
\mu_{i,s}^{prior}=\mu_{i,s-1}^{post}.
$$

At the first modeled transition, the 2022/23 opening mean is anchored to the 2021/22 unified strength index. Promoted clubs instead use the relevant promotion prior.

The model carries forward posterior means, not the entire previous covariance matrix. Opening covariance is rebuilt from the new season's team-specific prior variances. This allows past strength information to persist while setting a separate uncertainty scale for the season ahead.


### 27. Testing transfer spending in the prior mean


Summer expenditure and transfer income are tested as predictors of the change in unified strength:

$$
\Delta S_{i,s}
=\alpha+\beta_E E_{z,i,s}+\beta_I I_{z,i,s}+\varepsilon_{i,s}.
$$

Across 68 retained-team transitions, the expenditure coefficient is approximately 0.111, with $p\approx0.33$. The income coefficient is approximately −0.300, and the full model has $R^2=0.064$.

These fitted associations explain little of the observed variation. The more important decision is whether the financial adjustment improves prediction of later transitions, which is assessed next.


### 28. Transfer adjustments in chronological prediction


The expanding-window comparison uses $\widehat{\Delta S}=0$ as the reference: the team's existing strength is carried forward without a financial adjustment.

| Mean-transition specification | RMSE | MAE |
|---|---:|---:|
| No predicted change | 0.8091 | 0.6553 |
| Expenditure and income adjustment | 0.8249 | 0.6693 |
| Net-spend alternative | 0.8336 | — |

The expenditure/income version increases RMSE by about 1.95% and MAE by about 2.14%, using the unrounded results. The net-spend alternative also performs worse.

Aggregate transfer finance is therefore excluded from the final prior mean. This finding concerns the measures tested here; it does not establish that player arrivals and departures have no effect on football performance.


### 29. Testing transfer activity in prior uncertainty


The notebook also asks whether a more active transfer window predicts a less certain strength transition. A regression of squared transition residuals on transfer activity gives a coefficient of approximately 0.232, with $p=0.206$ and $R^2=0.034$.

This analysis supplies limited evidence for the proposed variance adjustment, so it is not included in Version 1. The result is a weak fitted association, rather than a separate demonstrated out-of-sample variance-model failure.


### 30. Using residual instability to set prior variance


The retained uncertainty mechanism uses the earlier instability index. Home and away values are combined on the variance scale:

$$
U_{i,s}^{2}
=\frac{U_{H,i,s}^{2}+U_{A,i,s}^{2}}{2},
\qquad
M_{i,s}
=\frac{U_{i,s}^{2}}
{\operatorname{Mean}_{j\in(\ell,s)}U_{j,s}^{2}}.
$$

Normalization is within the same league $\ell$ and season. Equal venue weights reflect the balanced schedules; the formula averages variation around the respective adjusted baselines.

The opening prior is

$$
\theta_{i,s}\sim
\mathcal N\!\left(\mu_{i,s}^{prior},Q_0M_{i,s-1}\right).
$$

Thus the prior mean summarizes retained strength information, while prior variance uses a shared fitted scale and relative residual instability. This is a modeling link between performance variability and strength uncertainty; the comparison with the reset model does not isolate its contribution on its own.


### 31. Updating from match results


The observation is the home team's goal difference:

$$
D_t=h_{\mathrm{home}}
+\beta(\theta_H-\theta_A)+\varepsilon_t,
\qquad
\varepsilon_t\sim\mathcal N(0,R).
$$

The Bayesian update uses this result rather than the full match-performance vector. Given the same prior and fixture, 2–0 and 3–1 therefore supply the same goal-difference observation.

The model maintains a full covariance matrix during each season. There is no added within-season process-noise term: posterior means change as beliefs are revised after results, rather than through a separately specified random walk in true strength.


### 32. Final recursive calibration


The final recursive calibration uses 1,520 EPL matches across 2022/23–2025/26. The fitted parameters are:

| Parameter | Estimate | Role |
|---|---:|---|
| $Q_0$ | 0.981691 | Opening variance scale |
| $h_{\mathrm{home}}$ | 0.282894 | Home advantage in goal difference |
| $\beta$ | 0.368582 | Strength-to-goal-difference coefficient |
| $R$ | 2.719278 | Observation-noise variance |

The sequential log-likelihood at this full-history fit is approximately −2958.248. This is a calibration result, not an independent performance test.

The earlier chronological evaluations use separate parameters fitted inside each historical window. In particular, the final $Q_0\approx0.982$ is reserved for subsequent application rather than substituted into past forecasts.


### 33. The season-reset comparison


The comparison model starts each club with a neutral mean and common variance at every season boundary:

$$
\boldsymbol\mu_0=\mathbf0,
\qquad P_0=Q_0I.
$$

It uses the same form of within-season Bayesian update. In the expanding-window evaluation, each model's global parameters are calibrated separately using its training seasons.

The comparison consequently evaluates the recursive prior construction as a whole against a season-reset system. It changes inherited means and team-specific opening variances, as well as allowing each specification its own calibration. The resulting gain should not be attributed exclusively to posterior-mean carryover without an additional ablation.


### 34. What the expanding-window evaluation found


The Bayesian evaluation fits on earlier seasons and predicts the next season, covering three target seasons and 1,140 matches:

| Model | Goal-difference RMSE | MAE | Predictive log-likelihood |
|---|---:|---:|---:|
| Recursive Bayesian | 1.687 | 1.329 | −2216.503 |
| Season reset | 1.725 | 1.346 | −2241.347 |

The recursive model has lower RMSE and higher predictive log-likelihood in each target season. Its advantage is largest in the early-season windows:

| Evaluation window | Recursive RMSE | Reset RMSE | Approximate RMSE reduction |
|---|---:|---:|---:|
| First five matches per team | 1.527 | 1.764 | 13.4% |
| First ten matches per team | 1.565 | 1.691 | 7.5% |
| Full target seasons | 1.687 | 1.725 | 2.2% |

This pattern is consistent with prior information being most useful when few current-season results are available. As results accumulate, the reset model can recover more of the strength ordering.

These are positive historical results for the Bayesian layer conditional on its upstream representation. They do not remove the need to refit all upstream transformations within a fully chronological end-to-end audit.


### 35. The opening state for 2026/27


Retained clubs start from their 2025/26 posterior means. Promoted clubs start from the final promotion bridge, and every club receives prior variance $Q_0M_i$. City's archived opening mean is approximately 3.026.

The Bayesian strength layer already covers the league. What is City-specific in Version 1 is the later performance-state analysis and score regression. The proposed league-wide extension would expand those layers, while building on an existing league-wide strength system.


## Part VII — City residual dynamics, structural scenarios, and scoring

*Source: dynamic_system_model(5).ipynb*


### 36. The question addressed by the City state


The City analysis asks whether a short-term performance deviation persists after accounting for team strength, opponent strength, and venue.

Its observed match vector is

$$
\mathbf x_t=
\begin{bmatrix}
PossDiff & ShotDiff & SOTRateDiff & FinishingDiff
\end{bmatrix}^{T}.
$$

For each dimension, the expected-performance baseline is fitted from strength difference and home status. The residual is

$$
\mathbf r_t=\mathbf x_t-\widehat E[\mathbf x_t\mid\Delta S_t,Home_t].
$$

Training-window residual means and standard deviations are then used to standardize the observations. These variables are available after a match; forecasting requires a pre-match prediction of the state, not the match's realized statistics.


### 37. Why the model examines residual persistence


Repeatedly strong raw performances can reflect a team's underlying ability or the opponents it faces. The residual question is more specific: after City performs above or below its fitted expectation, does that departure help predict the next match?

This operational definition of short-term form focuses on incremental information beyond the strength-and-venue baseline. It does not cover every possible meaning of form, and it does not imply that all other contextual influences have been removed.


### 38. The initial temporal checks


The exploratory lag-one correlations are small:

| Residual dimension | Lag-one correlation |
|---|---:|
| Possession difference | 0.056 |
| Shot difference | −0.172 |
| SOT-rate difference | −0.081 |
| Finishing difference | −0.101 |

The shot regression has a nominal HC3-robust $p$-value of 0.035, but explains only about 3% of variation. That isolated result, among several exploratory comparisons, is not strong evidence of a useful forecasting relationship.

These checks motivate direct predictive comparisons. They do not independently determine whether a dynamic model should be retained.


### 39. Comparing AR and VAR candidates


The candidate set includes no dynamics, separate diagonal AR equations, full VAR models, and Ridge-regularized VAR models with one or two lags.

On the common 36-target validation set in 2024/25, no dynamics achieves RMSE 0.9521 and Ridge VAR(2) achieves 0.9508. The reported improvement is only 0.142%, with a selected Ridge penalty of 1000. Full VAR(2) performs much worse, with validation RMSE 1.1712.

The small regularized gain warrants a later comparison rather than a strong claim of dynamic predictability. The unregularized result also shows the risk of estimating many lag coefficients from one season.


### 40. The later VAR comparison and its stability


After refitting on 2023/24 and 2024/25, the models are evaluated on the same 36 target matches in 2025/26:

| Model | RMSE |
|---|---:|
| No dynamics | 1.0616 |
| Ridge VAR(2), penalty 1000 | 1.0657 |

The dynamic model's reported relative gain is −0.3794%, meaning a small deterioration.

Its companion matrix has spectral radius approximately 0.1647, below one. The fitted recurrence is therefore asymptotically stable, but that numerical property does not provide a forecasting advantage. The heavily regularized model is also close to the no-dynamics reference in its predictions.


### 41. The decision on endogenous dynamics


The tested representation provides limited evidence of useful residual persistence. Version 1 therefore excludes the fitted VAR from the operational predictor while retaining it as a diagnostic result.

Information still passes between matches through Bayesian strength updates. The result only limits the additional pathway from one performance residual to the next; it does not establish that matches are independent or that all forms of football dynamics are uninformative.


### 42. The Maresca-era structural comparison


The coaching analysis uses Chelsea's Maresca-era performance as external evidence about a possible structural change. After adjustment and conversion to City residual-standard-deviation units, the estimated signature is approximately

$$
\mathbf d_M^{transfer}
=
\begin{bmatrix}
-0.686 & 0.294 & -0.369 & -0.260
\end{bmatrix}^{T},
$$

in the order possession, shots, SOT rate, and finishing. Possession has the clearest bootstrap evidence: its 95% interval excludes zero, while the other three intervals include zero.

A City-specific effect can be written conceptually as

$$
\mathbf b_M^{City}
=\mathbf d_M^{transfer}+\boldsymbol\delta_{\mathrm{transport}}.
$$

The transport term is not identified by the Chelsea sample. The vector therefore supplies external reference information for scenarios; it is not an estimated causal City effect or an automatic adjustment to score rates.


### 43. Representing midfield structure


The roster analysis initially focuses on midfield. Effects of changes elsewhere are not modeled separately, so any associated variation remains in the unexplained component.

Six SVD components are retained from sixteen standardized player features. The complete reference population contains 194 midfielder-seasons, and the retained components explain approximately 75.05% of reference-player variation.

This creates a common coordinate system for comparing player profiles. Its dimensions are interpreted from their feature loadings rather than assigned tactical labels in advance.


### 44. Comparing the two midfield pairings


The reference pairing combines Rodri's 2023/24 profile with Bernardo Silva's 2025/26 profile. It is compared with the 2025/26 profiles of Enzo Fernández and Elliot Anderson.

The standardized pair-center difference is approximately

$$
\Delta\mathbf p_{\mathrm{pair}}
=
\begin{bmatrix}
-0.570 & 1.737 & 0.437 & 0.443 & 0.094 & 0.182
\end{bmatrix}^{T}.
$$

Equal weights describe the composition of each pair. They do not assert equal future minutes, and the reference is a constructed comparison across seasons rather than an observed midfield from one season.


### 45. Mapping midfield profiles to team outcomes


The historical bridge relates minutes-weighted midfield profiles to team-season performance:

$$
\mathbf Y_{i,s}
=\boldsymbol\alpha+\boldsymbol\gamma S_{i,s}
+B_{\mathrm{mid}}\mathbf m_{i,s}
+\boldsymbol\epsilon_{i,s}.
$$

Here, $S_{i,s}$ is preseason strength, $\mathbf m_{i,s}$ has six components, and $\mathbf Y_{i,s}$ has four outcomes. The $4\times6$ matrix is a supervised association across these two levels of description.

The Extended role scope supplies 60 EPL team-seasons. The displayed validation search selects Ridge penalty $\lambda=30$, and the code uses that selected value. An intermediate Markdown paragraph still says $\lambda=10$; that is a documentation inconsistency, not the operative setting.


### 46. Calibrating and evaluating the midfield bridge


The full bridge does not help every outcome. A validation-derived blend shrinks its incremental predictions toward the strength-only baseline:

$$
\widehat{\mathbf Y}_{cal}
=\widehat{\mathbf Y}_{base}
+D_\rho
\left(\widehat{\mathbf Y}_{full}-\widehat{\mathbf Y}_{base}\right),
\qquad
\boldsymbol\rho=
\begin{bmatrix}
0.7460 & 0.2064 & 0.3316 & 0
\end{bmatrix}^{T}.
$$

The exploratory 2025/26 comparison is:

| Outcome | Strength-only RMSE | Full bridge RMSE | Calibrated RMSE | Calibrated reduction vs baseline |
|---|---:|---:|---:|---:|
| Possession difference | 0.4074 | 0.3910 | 0.3480 | 14.58% |
| Shot difference | 0.4637 | 0.4064 | 0.4274 | 7.83% |
| SOT-rate difference | 0.7569 | 0.7780 | 0.7498 | 0.94% |
| Finishing difference | 0.6096 | 0.6765 | 0.6096 | 0.00% |

Mean standardized RMSE falls from 0.5594 to 0.5337, approximately 4.6%. Most of the calibrated gain is in possession and shots, although the uncalibrated model has lower shot RMSE than the blend.

This is a contemporaneous historical association, with same-season player statistics and minutes and a retrospectively constructed SVD. It does not identify the effect of a transfer. Also, $\rho$ shrinks the whole incremental prediction, including changes in intercept and strength coefficients. Using $D_\rho B_{\mathrm{mid}}$ as a scenario matrix is a convenient approximation, not an independently identified reliability adjustment to each matrix row.


### 47. Moving from a pair to a team scenario


A two-player comparison does not represent the entire midfield. The scenario therefore approximates the team-profile shift as

$$
\Delta\mathbf m_{\mathrm{team}}
\approx s\,\Delta\mathbf p_{\mathrm{pair}},
\qquad s\in\{0.25,0.50,0.75\}.
$$

The values of $s$ make the assumed pair-minute share explicit. They do not estimate the future lineup or minutes of other midfielders.

Bridge outputs are also converted into compatible City units. In particular, the SOT-rate bridge output is multiplied by 100 to convert proportions to percentage points before division by City's residual standard deviation.


### 48. A conditional adaptation equation


The structural reference state is approached gradually:

$$
\mathbf d_{t+1}
=a_\tau\mathbf d_t+(1-a_\tau)\mathbf r+\boldsymbol\eta_t,
\qquad
a_\tau=2^{-1/\tau},
\qquad
A_\tau=a_\tau I.
$$

The half-life $\tau$ takes the scenario values 2, 5, or 10 matches. It is an assumption about adaptation speed, not a value estimated from historical coaching transitions. The diagonal form also assumes the same retention rate in all four dimensions without cross-component propagation.

For positive $\tau$, all eigenvalues lie between zero and one. Fixed-reference deterministic stability therefore follows from the chosen form. The Lyapunov and trajectory checks verify that the numerical implementation matches that specification. With continuing random disturbances, observed performances can still fluctuate.


### 49. The historical state-space comparison


The filter parameters are estimated on the first two City seasons, then used to predict the residuals in all 38 matches of 2025/26:

| Specification | Residual RMSE |
|---|---:|
| Zero-residual benchmark | 1.0466 |
| State space, half-life 2 | 1.0480 |
| State space, half-life 5 | 1.0513 |
| State space, half-life 10 | 1.0514 |

None improves this comparison. The historical replay uses a zero reference and records the state before absorbing the current match's observation. It tests filtered residual predictability, not the accuracy of a future coaching or midfield intervention scenario.

The adaptation model is retained as a conditional structural framework. These 38-match results are not directly comparable with the VAR's 36-target RMSE without aligning the evaluation sets and specifications.


### 50. Turning strength into score probabilities


The operational scorer uses pre-match strength difference:

$$
\Delta S_t=S_{\mathrm{City},t}^{pre}-S_{\mathrm{Opp},t}^{pre}.
$$

Separate regressions estimate City's goals for and goals against:

$$
\log\lambda_{GF,t}
=\alpha_{GF}+\beta_{GF}\Delta S_t+\gamma_{GF}Home_t,
$$

$$
\log\lambda_{GA,t}
=\alpha_{GA}+\beta_{GA}\Delta S_t+\gamma_{GA}Home_t.
$$

Given the fitted rates, $GF_t$ and $GA_t$ follow conditionally independent Poisson distributions. Their product defines the exact-score probabilities, which are summed to obtain win, draw, and loss probabilities.

The inputs are posterior mean strengths. Strength and coefficient uncertainty are not integrated into this probability distribution in Version 1.


### 51. Selecting and evaluating the score baseline


On 2024/25 validation data, the strength-and-venue model has mean score NLL 3.0992, compared with 3.2128 for venue only. It is selected for the score pipeline and refitted on the first two seasons.

The later 2025/26 comparison is mixed:

| Model | Mean score NLL | GF RMSE | GA RMSE |
|---|---:|---:|---:|
| Venue only | 2.9032 | 1.2360 | 0.9892 |
| Strength + venue | 2.9274 | 1.2285 | 1.0230 |

Lower values are better. Strength gives a small improvement in goals-for RMSE, but worse NLL and goals-against RMSE. Its win/draw/loss Brier score is also higher: 0.5501 versus 0.5396.

The validation-selected specification is retained, with those later results documented. It should be described as an operational baseline with mixed historical evidence, rather than a model shown to dominate venue only.


### 52. Adding the latent state to the scorer


The candidate extension adds pre-match filtered state features:

$$
\log\lambda_t
=\alpha+\beta\Delta S_t+\gamma Home_t
+\mathbf w^{T}\mathbf d_{t|t-1}.
$$

The experiment uses the five-match half-life, a zero structural reference, and the three dimensions with nonzero estimated state variation: possession, SOT rate, and finishing.

On 2025/26, mean score NLL rises from 2.9274 to 2.9910, an increase of approximately 2.17%. GF RMSE rises from 1.2285 to 1.2368, and GA RMSE from 1.0230 to 1.1046.

This exploratory comparison does not support adding the tested state features to Version 1. It is narrower than a test of all possible dynamics or of the future scoring consequences of a structural intervention.


### 53. Interpreting exact-score results


The strength-and-venue model includes the realized score in its Top 5 for 17 of 38 historical matches, or approximately 44.7%. The corresponding venue-only figure is 22 of 38.

Top 5 coverage depends on the full distribution and the probability mass assigned to those five scores. It should be considered alongside NLL, goal RMSE, win/draw/loss Brier score, and calibration. The selected model's mean Top 5 probability mass is approximately 44.48%; its numerical closeness to the observed hit rate is not, on its own, a calibration test.

The fitted distributions often favor moderate scores such as 1–1, 1–0, 2–1, and 2–0. Whether they systematically understate high-scoring outcomes or score dependence requires further diagnostics.


### 54. Refitting for 2026/27 use


After the historical comparisons, the selected score specification is refitted on all 114 City matches from 2023/24–2025/26. The resulting coefficients are approximately:

| Target | Intercept | Strength-difference coefficient | Home coefficient |
|---|---:|---:|---:|
| Goals for | 0.2356 | 0.1191 | 0.2663 |
| Goals against | 0.4097 | −0.1231 | −0.1276 |

The operational model uses the fitted coefficients at full precision. They remain fixed during the season, while Bayesian team-strength estimates update as results are entered.

Using completed historical matches for this prospective refit is appropriate. The refit does not supply another independent estimate of forecast quality.


### 55. The online workflow


The online module stores the preseason priors, Bayesian parameters, fixed score-model coefficients, result history, and issued forecasts.

Before a match, the user enters its date, opponent, and City's venue. The system constructs the current strength difference, generates probabilities, and saves the forecast. Afterward, the user enters the result, which updates strengths for subsequent predictions.

The stored configuration and match history allow the season to resume without resetting previous entries. Issued forecasts are preserved by the module rather than recomputed in place when later outcomes arrive. The actual update changes Bayesian beliefs, not the frozen Poisson coefficients.


### 56. Retrospective replay and prospective prediction


Some early 2026/27 rows were reconstructed after their matches had taken place. A replay uses only recorded results dated before the target fixture, so the target's own result is excluded from its strength input.

That chronological restriction makes the reconstruction useful for diagnosis. It does not turn a later calculation into a forecast issued before the match. The report distinguishes these sources explicitly, including forecasts saved on the match date whose relation to kickoff is not verified by a stored kickoff time.

In the saved five-match snapshot, two reconstructed scores appear in the Top 5. This is a descriptive replay result from a very small sample, not a prospective performance claim.


## Part VIII — Decisions across the project


### 57. What each component contributes


| Component | Version 1 decision | Evidence or role |
|---|---|---|
| Difference-based performance features | Retain | Separate interpretable aspects of match performance |
| Linear opponent adjustment | Retain | Supported associations, especially for shots and corners |
| Quadratic opponent response | Exclude | No gain meeting the stated CV threshold |
| Team-specific opponent slopes | Exclude from final adjustment | Negligible hierarchical-model CV gain |
| Team-specific baselines | Retain | Represent differences in venue-adjusted performance |
| Residual instability $U$ | Retain | Supplies the relative prior-variance multiplier |
| Unequal strength weights | Retain | Modest held-out-season ranking gain and similar multi-seed patterns |
| Competitive tiers | Interpretation only | Continuous strength remains the model input |
| PCA PC1 | Retain | Main shared home-away component, with strong ClubElo agreement |
| PCA PC2 | Retain separately | Describes venue asymmetry; not a final score-rate input |
| ClubElo | External comparison only | Checks the interpretation of the index |
| Team-specific Championship bridge | Exclude | Unstable or worse cohort-validation performance |
| Common promotion mean | Retain | Better supported than the tested linear bridge |
| Investment in the promotion bridge | Exclude | No cohort-validation gain |
| Transfer finance in retained-team prior means | Exclude | Worse expanding-window prediction |
| Transfer activity in prior variance | Exclude | Limited fitted evidence |
| Recursive Bayesian prior | Retain | Better historical RMSE and log-likelihood than the reset system |
| City VAR dynamics | Diagnostic only | No later residual-prediction gain |
| Maresca-era signature | Structural reference | Observational evidence with unresolved transport uncertainty |
| Midfield SVD bridge | Exploratory structural analysis | Some calibrated association, without strict preseason validation |
| Conditional adaptation | Scenario framework | Stable assumed dynamics; historical filter gives no predictive gain |
| Latent-state score extension | Exclude | Worse historical NLL and goal RMSE |
| Independent Poisson scorer | Retain as baseline | Produces usable probabilities, with mixed comparative results |


### 58. The clearest positive comparison


The recursive Bayesian prior provides one of the project's clearest positive historical comparisons. Its advantage over resetting every season is most pronounced when few new results are available, then narrows as both models learn from the current season.

This supports continued use of the recursive system. The result concerns the full prior specification and its calibration; a separate experiment would be needed to divide the gain between inherited means and instability-based variances.


### 59. The repeated negative finding


The tested City residual dynamics do not show a convincing incremental forecasting benefit. The evidence includes weak exploratory lag relationships, a very small VAR validation gain that reverses in the later comparison, state filters that lose to zero residual, and a latent-state extension that worsens score NLL.

These checks address related but different questions and reuse overlapping data. They should not be counted as independent replications. Together, they justify leaving those particular extensions out of Version 1 while preserving the question for new data and designs.


### 60. How the simpler forecaster still learns


A match result changes the Bayesian strength distribution. The revised means then change the strength difference and goal rates used for a later fixture.

The operational forecaster therefore carries information across matches even though it has fixed Poisson coefficients. What it omits is a separate predictive recurrence for the four performance residuals.

Conditional independence between the two goal counts in a fixture is also a different assumption from treating successive forecasts as unrelated. The strength-update process links those forecasts through their shared history.


## Part IX — Limits of the current evidence


### 61. Limited City data

The City-only sample is intentionally restricted to recent seasons rather than expanded as far backward as possible. The purpose of the project is to model current and near-future team behavior, and football clubs are strongly nonstationary over long horizons. Coaching staff, squad composition, tactical structure, competitive level, and even the league environment can change substantially over a decade. Adding much older observations would increase sample size, but it could also mix materially different data-generating regimes.

The chosen window therefore reflects a deliberate trade-off between sample size and temporal comparability. More observations are not automatically more informative if they describe a substantially different version of the team. The project prioritizes recent-season relevance and controls model complexity accordingly rather than maximizing historical depth.
This choice still has a statistical cost. One City league season contains only 38 matches, or 36 usable targets for a within-season two-lag model.One City league season contains 38 matches, or 36 usable targets for a within-season two-lag model. A four-dimensional VAR(2) has 32 lag coefficients and four intercepts: eight lag coefficients plus an intercept in each output equation.

The estimation problem is therefore demanding relative to the initial training sample. Weak dynamics and poorly estimated dynamics are difficult to distinguish. The observed forecasting results support excluding the tested models under the available data, but do not establish that no useful residual state exists.


### 62. Repeated use of 2025/26


The 2025/26 season initially served as a later evaluation period. Further structural and state-space experiments were developed after earlier results from that season had been inspected.

Model-development decisions can adapt to those outcomes even when a particular fitting function excludes them. This creates a risk of optimistic model selection through repeated test reuse. A comparison fixed before inspecting its targets has a stronger claim to independence than a later variant motivated by those same outcomes.

Later 2025/26 results are therefore reported as exploratory. Upstream preprocessing also needs its own timing audit; avoiding direct target input in the final predictor is not sufficient to establish an untouched end-to-end test.


### 63. Using completed history for future forecasts


Once 2025/26 is complete, its data can legitimately contribute to a model intended for later fixtures. The problem with adaptive test reuse concerns the strength of historical evaluation claims, not permission to learn from completed matches.

The full-history 2026/27 refit is a reasonable operational step. Its predictive quality remains an empirical question to be assessed using forecasts preserved before subsequent results become known.


### 64. The timing of midfield information


The bridge relates realized player statistics and minutes to team outcomes from the same season. Those inputs are unavailable at the start of that season, and common tactical influences can affect both sides of the relationship.

A strict preseason version would use prior-season profiles, known roster information, and explicit forecasts of roles and minutes. Scaling, imputation, and SVD would be fitted using only the earlier training data and then applied to the later period.

That redesign would test a different and more demanding claim: whether information available in advance predicts subsequent team performance.


### 65. What the coaching comparison can identify


The Chelsea comparison controls for strength difference and venue, but cannot fully separate coaching from simultaneous changes in the squad, availability, player development, or other tactical and contextual factors.

Its estimated vector describes a coaching-era association. Applying it to City introduces further uncertainty because the clubs have different starting styles and personnel. Resampling Chelsea's historical observations quantifies uncertainty in that calculation, not all of the uncertainty in transferring it to another club.


### 66. Assumptions of the score distribution


The independent Poisson model imposes conditional independence and Poisson variation at each fitted rate. It also plugs in posterior mean strengths and fitted coefficients without propagating their uncertainty into the final distribution.

Potential problems include remaining goal dependence, conditional overdispersion, and poor probability calibration in some score ranges. These possibilities need diagnostics; a few surprising scorelines alone do not identify which assumption is responsible.

Future alternatives should be compared on the same fixtures and information cutoffs using proper probability scores, alongside goal error and calibration measures.


## Part X — The status of Version 1


### 67. What is complete


Version 1 has a connected modeling workflow: adjusted team states, a unified strength coordinate, promotion initialization, recursive Bayesian updates, comparisons of dynamic alternatives, structural scenarios, score distributions, and a persistent online prediction record.

The first modeling cycle is complete in that practical sense. Completion does not imply that every module improves prediction, that every validation is fully prospective, or that the selected score model is optimal. Those qualifications are part of the result.

The next operational task is to use the frozen forecaster consistently and evaluate the predictions it actually issued.


## Part XI — Near-term improvements


### 68. Create a prospective forecast record


Before each future match, record the information cutoff, model version, pre-match goal rates, outcome probabilities, and ranked score predictions. Retain enough information to reconstruct the complete score distribution, and preserve the issue time.

The present date-based workflow uses results from earlier dates. A future version using same-day information should also record reliable kickoff and result times. After a match, record its outcome and update the state used for later fixtures without replacing the original forecast.

As the sample grows, evaluate score NLL, GF and GA RMSE, win/draw/loss Brier score, probability calibration, and Top 1/3/5 coverage. Keep retrospectively reconstructed predictions in a separate evaluation category.


### 69. Keep league results current


The existing Bayesian layer can already accept results from all EPL clubs. In the saved online snapshot, however, all five recorded results involve City.

Entering other completed league matches would allow opponent strengths to absorb evidence from those fixtures. This improves information coverage without changing the score-regression structure. It is an operational improvement whose effect on forecast accuracy can then be measured.


### 70. Audit the full chronological pipeline


A stronger evaluation should fit each learned transformation inside the appropriate historical window. This includes the original strength weights and scaling, PCA/SVD, imputation rules, promotion mapping, Bayesian calibration, residual baselines, state noise, and score regressions.

Model forms and tuning values should be chosen using earlier validation periods. The later assessment period should remain outside those decisions until the pipeline is fixed.

This audit is broader than rerunning the final regression with a date filter. It checks when every input and learned representation would actually have been available.


### 71. Use comparable external benchmarks


Future forecasts can be compared with external systems when predictions are available for the same fixtures and before the same information cutoff.

The metric should match the published output. A system that supplies only a most-likely score cannot support an exact-score NLL comparison; probability scores require the corresponding probabilities. Win/draw/loss distributions can be compared with Brier scores, and goal-rate estimates with goal RMSE.

Comparisons against unrelated league-wide average hit rates would mix differences in fixture difficulty, forecast timing, and evaluation samples.


## Part XII — A possible league-wide Version 2


### 72. The proposed extension


Version 2 would extend the City performance-state and score models to the whole Premier League. The Bayesian strength layer already estimates a value for each club; the proposed addition is a consistent performance-state representation for every team.

A fixture would then be described by two evolving pairs,

$$
(\theta_i,\mathbf z_i)
\quad\text{and}\quad
(\theta_j,\mathbf z_j),
$$

where $\theta$ is strength and $\mathbf z$ is a pre-match residual performance state. This is a proposed architecture, not an implemented forecasting result.


### 73. How opponent states might help


A City-only performance model captures little direct information about the opponent's recent residual performance. A league-wide version could make that information available on both sides of the fixture.

For example, a future goal-rate function could take the form

$$
\lambda_i
=f\!\left(\theta_i-\theta_j,\mathbf z_i,\mathbf z_j,Home\right).
$$

The function would need to be estimated and evaluated. Adding opponent states could help if they contain information beyond strength, but could also add noise or duplicate information already represented.


### 74. Sharing information across clubs

A league-wide extension is attractive partly because it increases the effective information set cross-sectionally rather than by reaching much further backward in time. This preserves the preference for recent, more comparable football regimes while reducing the small-sample problem.

Fitting twenty unrelated VAR models would retain the same small-sample difficulty for each club. A hierarchical or panel model is a candidate way to share information:

$$
\mathbf z_{i,t+1}
=\left(A_{\mathrm{global}}+\Delta A_i\right)\mathbf z_{i,t}
+\boldsymbol\eta_{i,t}.
$$

The shared matrix captures common behavior, while regularized team deviations allow supported differences. Weakly estimated deviations would shrink toward the common pattern.

This structure has its own assumptions. Clubs may differ in ways that are poorly represented by one shared matrix, and each fixture supplies linked observations for two teams. A league-wide dataset therefore does not provide two independent matches simply because it has two team-perspective rows.


### 75. A candidate Version 2 workflow


The proposed workflow has six stages:

1. Update league-wide Bayesian strengths using completed results.
2. Estimate expected performance for each club given strength, opponent, and venue.
3. Construct comparable residual observations for both sides of each fixture.
4. Estimate shared and team-specific dynamics, recording only pre-match state forecasts.
5. Use both teams' strengths and states to estimate goal rates.
6. Evaluate the resulting score distributions against a matched no-state baseline.

Player and coaching scenarios could be considered later, with information timing and uncertainty made explicit. The essential first test is whether the shared state model contributes predictive information under a controlled chronological comparison.


### 76. What the league-wide experiment could establish


Pooling clubs could make it easier to estimate relationships that are too noisy in the City-only sample. A gain over a carefully matched no-dynamics baseline would support that particular pooled specification.

It would not by itself prove that sample size caused the Version 1 result: the extension also changes the model structure and introduces opponent states. Comparisons that vary pooling, opponent information, and dynamics separately would help distinguish those contributions.

A failure would narrow the evidence against those tested specifications. It would still not establish a universal absence of useful football dynamics.


### 77. Treating improvement as a hypothesis


A league-wide model could provide more observations and a fuller description of each opponent. Whether that produces better forecasts depends on the quality of the variables, the pooling assumptions, estimation error, and validation design.

The Version 2 objective should therefore be stated as a question: can shared estimation and opponent performance states improve forecasts beyond the existing strength baseline? The answer must come from new comparisons, rather than from the size or complexity of the proposed model.


## Part XIII — Reflection on the full project


### 78. The initial expectation


The project began by trying to combine Bayesian inference, dynamic systems, coaching changes, and midfield reconstruction in a City score predictor. The initial expectation was that each layer would add useful information and move the model closer to a football explanation.

The experiments changed that expectation. They showed that different components needed different roles, and that some would remain descriptive even after being implemented mathematically.


### 79. The outcome of the first modeling cycle


The recursive strength system produced a useful positive historical comparison. The short-term residual extensions did not show comparable gains, and the structural modules retained substantial interpretive uncertainty.

The resulting score model is simpler than the initial concept, but it sits within a broader record of tested assumptions and alternatives. The project now provides both a working forecast process and evidence about which additions were not supported under the current design.

Its limits remain consequential: the final scorer has mixed historical performance against venue only, and a genuine prospective archive is still needed.


### 80. Lessons to carry into the next version


| Lesson | What the project shows |
|---|---|
| Give football hypotheses an empirical test | Plausible transfer and dynamic features did not consistently improve the relevant comparisons. |
| Separate stability from prediction | A recurrence can be stable while providing no useful forecast gain. |
| Distinguish strength memory from residual form | The recursive prior and short-term residual models address different sources of information. |
| Match each claim to its evidence | Structural associations, cohort validation, chronological replay, and prospective forecasts support different conclusions. |
| Preserve useful analysis outside the scorer | Coaching and midfield modules can describe conditional scenarios without being automatic goal-rate inputs. |
| Improve information and evaluation before adding complexity | Broader data may help, but better timing controls, appropriate pooling, and comparable benchmarks are also necessary. |

One possible explanation for the weak residual gains is overlap with information already absorbed by Bayesian strength. Noise, sample size, and model specification are other plausible explanations. The project has not identified their relative contributions.


### 81. The two parts of the final architecture


| Layer | Inputs and processing | Current output |
|---|---|---|
| Operational forecasting | Completed results update league strengths; City's pre-match strength difference and venue determine two Poisson goal rates. | Score and outcome probabilities, saved before the match and compared with its later result |
| Structural analysis | Coaching-era evidence and midfield profiles define conditional reference shifts and adaptation assumptions. | Scenario trajectories and diagnostics, with estimation and transport limits stated |

The structural analysis is not silently added to the operational rates. Any future connection should be specified and tested explicitly before it becomes part of the issued forecast.


### 82. Where the project stands


Version 1 reaches a practical stopping point: the main pipeline is implemented, alternatives have been compared, and the online workflow can preserve forecasts and update strengths as new results arrive.

The next evidence should come from using that system consistently. A later league-wide state model is a reasonable research direction, provided that its added information and pooling assumptions are tested against the same strength baseline.

The central question for Version 2 is whether shared estimation and better opponent representation can produce a measurable gain on genuinely later matches. That question remains open.


## Appendix A — Source notebooks and their roles

| File | Main role |
|---|---|
| *team_strength_final(1).ipynb* | Opponent-adjusted multivariate team states and residual instability |
| *strength_index_validation(1).ipynb* | Venue-specific scalar strength and weight comparisons |
| *strength_index_pca_elo_validation(1).ipynb* | Home-away PCA and external agreement with ClubElo |
| *E0_E1_bridge(1).ipynb* | Promotion initialization on opponent-strength and unified-strength scales |
| *bayesian_prior(1).ipynb* | Recursive priors, league-wide Bayesian updates, and season-reset comparisons |
| *dynamic_system_model(5).ipynb* | City residual dynamics, structural scenarios, score generation, and online use |

The section numbers in this retrospective describe its own 82 topics. References to numbered sections inside a source notebook refer to that notebook's headings.


## Appendix B — Version 1 status and immediate priorities

**Operational model.** Bayesian strength difference and venue feed an independent Poisson score distribution. Score-regression coefficients are fixed during the season; strength estimates update from recorded results.

**Structural modules.** The project retains the Maresca-era signature, midfield SVD and team bridge, and conditional adaptation scenarios for interpretation.

**Extensions excluded from operational rates.** The City VAR, historical residual filter, latent-state score extension, and aggregate transfer-finance adjustments have not been retained as additional production predictors.

**Immediate priorities**

1. Preserve genuinely prospective predictions and their information cutoffs.
2. Enter completed results from the whole EPL into the existing strength layer.
3. Audit the full pipeline under chronological fitting and model selection.
4. Compare forecasts with suitable benchmarks on identical fixtures.
5. Treat a league-wide hierarchical state model as a separate, testable Version 2 proposal.
